# Brent-WTI Hedge Ratio During Crude-Oil Event Windows

This notebook summarizes the hedge-ratio robustness check on short periods when crude oil prices moved sharply. Each window uses Databento `GLBX.MDP3` continuous futures, 1-minute bars, for `CL.v.0` and `BZ.v.0`.

**Question:** do event windows justify replacing the static 1:1 Brent-WTI spread with a fitted hedge ratio?

**Answer:** no. The evidence still supports using the simple `CL - BZ` spread as the primary strategy input.

## Method

For each six-day window:

1. Pull `ohlcv-1m` bars for continuous front-month `CL.v.0` and `BZ.v.0`.
2. Align both contracts on the 1-minute timestamp.
3. Construct the static spread: `CL close - BZ close`.
4. Estimate a level regression: `CL = alpha + beta * BZ`.
5. Estimate a return regression: `diff(CL) = alpha + beta * diff(BZ)`.
6. Compare the fitted residual volatility against the simple 1:1 spread.

In [ ]:
from pathlib import Path
import os
import tempfile

os.environ.setdefault("MPLCONFIGDIR", tempfile.mkdtemp(prefix="mplconfig-"))

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "_output").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_PATH = PROJECT_ROOT / "_output" / "event_windows" / "hedge_ratio_event_windows.csv"
DATA_DIR = PROJECT_ROOT / "_data" / "event_windows"

results = pd.read_csv(RESULTS_PATH)
results

Matplotlib is building the font cache; this may take a moment.


## Windows Tested

These windows were chosen to capture crude-oil stress or repricing regimes: COVID shock, reopening/vaccine repricing, Russia-Ukraine invasion, Iran-Israel escalation periods, plus one calmer baseline week without a major new global conflict shock.

In [ ]:
window_table = results[["description", "start", "end_exclusive", "rows", "cl_start", "cl_end", "bz_start", "bz_end"]].copy()
window_table[["cl_start", "cl_end", "bz_start", "bz_end"]] = window_table[["cl_start", "cl_end", "bz_start", "bz_end"]].round(2)
window_table

## Hedge-Ratio Diagnostics

The most relevant number for sizing a market-neutral futures spread is the **return beta**, because it measures co-movement in price changes. A return beta near 1 supports the static 1:1 construction.

The level beta is useful as a fair-value regression check, but a level fit can be unstable in short event windows and can look better in-sample without improving the tradable spread.

In [ ]:
diagnostics = results[[
    "description",
    "return_beta",
    "level_beta",
    "return_r2",
    "level_r2",
    "corr_returns",
    "corr_levels",
    "spread_std",
    "ols_resid_std",
    "ols_std_improvement_vs_1to1",
]].copy()

round_cols = diagnostics.columns.drop("description")
diagnostics[round_cols] = diagnostics[round_cols].round(4)
diagnostics

## Visual Summary

The key checks are: return beta near 1, level beta stability, and whether a fitted level regression meaningfully reduces residual volatility versus the simple `CL - BZ` spread.

In [ ]:
plot_df = results.copy()
plot_df["label"] = plot_df["description"].str.replace(" / ", "\n", regex=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)

axes[0].barh(plot_df["label"], plot_df["return_beta"], color="#2F6F73")
axes[0].axvline(1.0, color="black", linewidth=1, linestyle="--")
axes[0].set_title("Return Hedge Ratio")
axes[0].set_xlabel("beta")

axes[1].barh(plot_df["label"], plot_df["level_beta"], color="#8A5A44")
axes[1].axvline(1.0, color="black", linewidth=1, linestyle="--")
axes[1].set_title("Level Hedge Ratio")
axes[1].set_xlabel("beta")

axes[2].barh(plot_df["label"], plot_df["ols_std_improvement_vs_1to1"], color="#5D6C89")
axes[2].axvline(0.0, color="black", linewidth=1)
axes[2].set_title("OLS Volatility Improvement")
axes[2].set_xlabel("fraction of 1:1 spread std")

for ax in axes:
    ax.grid(axis="x", alpha=0.25)
    ax.set_ylabel("")

plt.show()

## Static 1:1 Spread Paths

These plots show the simple `CL - BZ` spread inside each stress window. The large COVID negative-WTI episode is visibly different from the later windows, but the fitted hedge-ratio tests still do not provide strong evidence for replacing the 1:1 construction.

In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(12, 11), sharex=False, constrained_layout=True)

for ax, row in zip(axes, results.itertuples(index=False)):
    path = DATA_DIR / f"{row.window}_cl_bz_1m_{row.start}_{row.end_exclusive}.parquet"
    df = pd.read_parquet(path)
    df["synth_mid"].plot(ax=ax, color="#2F6F73", linewidth=1)
    ax.axhline(df["synth_mid"].mean(), color="black", linestyle="--", linewidth=1)
    ax.set_title(row.description)
    ax.set_ylabel("CL - BZ")
    ax.grid(alpha=0.25)

plt.show()

## Conclusion

The event-window results support keeping the Brent-WTI strategy on a static **1:1 spread**.

- Return betas are mostly close to 1, ranging from roughly 0.87 to 0.98 across the selected windows.
- Level betas move around more, especially in short crisis samples, which makes them less attractive as a production hedge ratio.
- In most windows, OLS residuals reduce in-sample spread volatility by very little versus the 1:1 spread.
- The non-conflict baseline has a larger in-sample OLS volatility improvement, but that is one short sample and does not outweigh the simpler futures contract logic of `CL - BZ`.

**Final decision:** use `CL - BZ` as the main spread. Treat fitted hedge ratios as robustness diagnostics, not as the primary construction.